# 1 · Data and measures

This page builds every variable used in the analysis, one step at a
time, starting from the children’s decisions in the game and their
answers to the nomination survey:

| Step | Measure | What it captures |
|------------------------|------------------------|------------------------|
| 1–2 | Cooperation GAP | who came out ahead in a pair’s exchange |
| 3 | Friendship type | whether the two children named each other as best friends |
| 4 | Social Rank | each child’s standing in the classroom, from popularity nominations |
| 5 | Social Rank GAP | the rank difference between the two children of a pair |
| 6 | Null-model filter | a robustness check that keeps the most partner-specific transfers |

The code shown here is the code in `code/01_build_measures.R`: the page
reads it chunk by chunk (with `knitr::read_chunk`), so the script and
the page can never drift apart. Running this page also writes the two
files used by the [main results](02_main_results.ipynb).

# The data

The experiment took place in 45 classrooms. In each classroom, children
were paired with every classmate once, across several rounds. In each
pair, the child received 10 tokens and decided how many to give to the
partner (tokens given were doubled for the receiver). After the game,
each child answered nine peer-nomination questions (name 2 to 5
classmates who…).

There are two files:

-   `data/raw/dyads.csv`: one row per **decision**, sender $i$ →
    receiver $j$. It has the tokens $i$ sent to $j$, the round, and nine
    0/1 columns `nom_q1` to `nom_q9` that say whether $i$ named $j$ in
    each question.
-   `data/raw/students.csv`: one row per child, with sex, age, GPA and
    attendance.

The data use study codes. Variable definitions are in
`data/codebook.md`.

In [ ]:
here::i_am("notebooks/main/01_data_and_measures.qmd")
library(tidyverse)
library(igraph)
library(here)

In [ ]:
dyads    <- read_csv(here("data", "raw", "dyads.csv"),    show_col_types = FALSE)
students <- read_csv(here("data", "raw", "students.csv"), show_col_types = FALSE)

In [ ]:
cat("Directed decisions:", format(nrow(dyads), big.mark = ","), "\n")   # 17,804
## Directed decisions: 17,804
cat("Children:          ", nrow(students), "\n")                         # 859
## Children:           859
cat("Classrooms:        ", n_distinct(dyads$class_id), "\n")             # 45
## Classrooms:         45

Head of dataframe with game (dyad) data

In [ ]:
head(dyads)

class_id,round,sender_id,receiver_id,tokens_sent,seconds_since_start,nom_q1,nom_q2,nom_q3,nom_q4,nom_q5,nom_q6,nom_q7,nom_q8,nom_q9
1,1,30,328,1,24,0,0,0,0,0,0,0,0,0
1,1,68,230,4,19,0,0,0,0,0,0,0,0,0
1,1,125,161,3,2,0,0,0,0,0,0,0,0,0
1,1,156,319,5,25,0,0,0,0,0,0,0,0,0
1,1,161,125,2,8,0,0,0,0,0,0,0,0,0
1,1,224,711,1,14,0,0,0,0,0,0,0,0,0


# Step 1. Tokens received

A decision tells us what $i$ sent to $j$. What $i$ *received* from $j$
is $j$’s decision toward $i$, which is another row of the same file. We
find it by swapping sender and receiver and joining.

In [ ]:
# Each row is one decision: sender i gives `tokens_sent` (0 to 10) to receiver j.
# In the same round, j also decided how much to give to i. That reverse
# decision is what i receives. We find it by swapping sender and receiver.

reverse_decisions <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         tokens_received = tokens_sent,
         round_reverse = round)

dyads <- dyads %>%
  left_join(reverse_decisions, by = c("class_id", "sender_id", "receiver_id"))

# every decision has its reverse, and both happened in the same round
stopifnot(!anyNA(dyads$tokens_received))
stopifnot(all(dyads$round == dyads$round_reverse))

dyads <- dyads %>% select(-round_reverse)

Both decisions of a pair are always in the same round: each pair of
classmates played together once, and in that round each one decided for
the other. We can also check directly that no pair played more than
once:

In [ ]:
dyads %>%
  mutate(child_a = pmin(sender_id, receiver_id),
         child_b = pmax(sender_id, receiver_id)) %>%
  group_by(child_a, child_b) %>%
  summarise(rounds = n_distinct(round), decisions = n(), .groups = "drop") %>%
  count(rounds, decisions)

rounds,decisions,n
1,2,8902


8,902 pairs, each with two decisions in a single round.

# Step 2. Cooperation GAP

$$
\text{Coop GAP}_{ij} = w_{ji} - w_{ij} \quad \text{(tokens received minus tokens sent)}
$$

In [ ]:
# Cooperation GAP (from the sender's point of view): what i receives minus what
# i sends. Positive values mean i receives more than i gives.

dyads <- dyads %>%
  mutate(coop_gap = tokens_received - tokens_sent)

In [ ]:
summary(dyads$coop_gap)
##    Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
##     -10      -3       0       0       3      10

The Cooperation GAP of $j \to i$ is the Cooperation GAP of $i \to j$
with the opposite sign, so it averages exactly zero over the whole
sample. What matters is not its level but *who* ends up ahead: the
analysis asks whether the child who comes out ahead tends to be the
higher-ranked one.

# Step 3. Friendship type

Question 3 asked for the child’s **best friends**. A pair is *mutual* if
both named each other, *one-sided* if only one did, and *non-friends* if
neither did. For one-sided pairs we keep track of who named whom.

In [ ]:
# Question 3: "Select 2-5 classmates you consider your best friends".
# nom_q3 = 1 if the sender named the receiver. To know if the receiver also
# named the sender, we look at the reverse row, as we did with the tokens.

reverse_friend <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         friend_reverse = nom_q3)

dyads <- dyads %>%
  left_join(reverse_friend, by = c("class_id", "sender_id", "receiver_id")) %>%
  mutate(friendship = case_when(
    nom_q3 == 1 & friend_reverse == 1 ~ "Mutual",
    nom_q3 == 1 & friend_reverse == 0 ~ "Sender declares",    # one-sided, i named j
    nom_q3 == 0 & friend_reverse == 1 ~ "Receiver declares",  # one-sided, j named i
    nom_q3 == 0 & friend_reverse == 0 ~ "Non-friends"
  ))

cat("Decisions by friendship type:\n")
## Decisions by friendship type:
dyads %>% count(friendship) %>% mutate(percent = round(100 * n / sum(n), 2))

friendship,n,percent
Mutual,1288,7.23
Non-friends,13916,78.16
Receiver declares,1300,7.30
Sender declares,1300,7.30


Each one-sided pair appears twice: once as “Sender declares” (seen from
the child who named the other) and once as “Receiver declares” (seen
from the other child). Together they are the 14.6% of asymmetric
friendships.

# Step 4. Social Rank

Social Rank comes from questions 4 (*most popular and well-known*) and 5
(*not popular or well-known*). We combine them in one network per
classroom, where the edge $i \to j$ has weight

$$
a_{ij} = P^+_{ij} - P^-_{ij} + 1 \in \{0, 1, 2\}.
$$

In [ ]:
# Questions 4 and 5: "most popular and well-known" and "not popular or
# well-known". For every ordered pair (i, j) in a classroom:
#
#     s_ij = nom_q4 - nom_q5      +1 popular, -1 not popular, 0 not named
#     a_ij = s_ij + 1             weights 2, 0 and 1 (all non-negative)
#
# a_ij is the weight of the edge i -> j in the popularity network.

popularity_edges <- dyads %>%
  transmute(class_id,
            from = as.character(sender_id),
            to   = as.character(receiver_id),
            weight = nom_q4 - nom_q5 + 1)

cat("Edges of the popularity network by weight (2 popular, 0 not popular, 1 not named):\n")
## Edges of the popularity network by weight (2 popular, 0 not popular, 1 not named):
count(popularity_edges, weight)

weight,n
0,1788
1,13868
2,2148


The $+1$ keeps every weight non-negative, as PageRank requires: a
popularity nomination gives weight 2, no nomination weight 1, and a
non-popularity nomination weight 0.

Why PageRank rather than a simple count of nominations? PageRank gives
more weight to nominations that come from children who are themselves
highly regarded, in the same way that a web page linked by important
pages is important. A child named as popular by well-regarded classmates
ranks higher than one named by classmates whom few regard as popular.

We compute PageRank in each classroom and rescale it from 0 (lowest in
the class) to 1 (highest in the class), so that ranks are comparable
across classrooms of different size.

In [ ]:
# PageRank (damping 0.85) is computed separately in each classroom.
# We then rescale it within each classroom to go from 0 (lowest rank in the
# class) to 1 (highest rank in the class). This is our Social Rank.

pagerank_one_class <- function(edges_one_class) {
  network <- graph_from_data_frame(edges_one_class %>% select(from, to, weight),
                                   directed = TRUE)
  pr <- page_rank(network, damping = 0.85, directed = TRUE)$vector   # uses edge weights
  tibble(student_id = as.numeric(names(pr)), pagerank = pr)
}

social_rank <- popularity_edges %>%
  group_by(class_id) %>%
  group_modify(~ pagerank_one_class(.x)) %>%
  mutate(social_rank = (pagerank - min(pagerank)) / (max(pagerank) - min(pagerank))) %>%
  ungroup()

students <- students %>%
  left_join(social_rank %>% select(student_id, pagerank, social_rank),
            by = "student_id")

cat("Social Rank (0 to 1 within each classroom):\n")
## Social Rank (0 to 1 within each classroom):
summary(social_rank$social_rank)
##    Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
##  0.0000  0.2742  0.4324  0.4580  0.6261  1.0000

As an illustration, the ranks in the small classroom used in Figure 2:

In [ ]:
students %>%
  filter(class_id == 16) %>%
  select(student_id, female, pagerank, social_rank) %>%
  arrange(desc(social_rank))

student_id,female,pagerank,social_rank
657,0,0.1590302,1.0000000
505,1,0.1282099,0.5930586
546,1,0.1252742,0.5542959
687,1,0.1080575,0.3269715
305,1,0.1056854,0.2956514
257,0,0.1022316,0.2500477
662,0,0.0983122,0.1982968
176,1,0.0899051,0.0872924
808,1,0.0832939,0.0000000


# Step 5. Social Rank GAP

$$
\text{RankGAP}_{ij} = \text{Rank}_i - \text{Rank}_j
$$

Positive values mean the sender has the higher rank. Since both ranks
run from 0 to 1 within a classroom, the gap runs from $-1$ (the
lowest-ranked child meets the highest-ranked one) to $1$ (the reverse).
Most pairs are much closer: 83% of the decisions have a gap between
$-0.5$ and $0.5$.

In [ ]:
# Social Rank GAP of a decision = sender's rank - receiver's rank.
# Positive: the sender has higher rank. It goes from -1 to 1.

dyads <- dyads %>%
  left_join(social_rank %>% select(sender_id = student_id, sender_rank = social_rank),
            by = "sender_id") %>%
  left_join(social_rank %>% select(receiver_id = student_id, receiver_rank = social_rank),
            by = "receiver_id") %>%
  mutate(rank_gap = sender_rank - receiver_rank,
         # round away floating-point noise, so that two children with the
         # same rank have a gap of exactly 0 (and not 1e-16)
         rank_gap = round(rank_gap, 10))

cat("Social Rank GAP (sender - receiver):\n")
## Social Rank GAP (sender - receiver):
summary(dyads$rank_gap)
##    Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
## -1.0000 -0.2211  0.0000  0.0000  0.2211  1.0000

# Step 6. Student covariates

Some analyses use student characteristics: attendance, GPA and age, and
the nominations each child received in each peer question. All of them
are standardized here, within each classroom (mean 0 and SD 1 among
classmates), so a value compares a child with her own classmates and not
with children of other schools. The z-scores are saved in
`analysis_students.csv`, and every script reads them from there.

-   `attendance_z` enters the balance check (Figure 1D, Table S16,
    Figures S15–S17). GPA enters that check on its original 1–7 scale,
    as in the paper.
-   The other z-scores are used in the correlates of Social Rank (Table
    S9, Figure S8) and of Behavioral Social Rank (Figure S30).
-   The two computed ages above 15 years are birth-date errors (SI Table
    S1) and are set to missing before standardizing.

In [ ]:
# Every standardized covariate used in the paper is built here, so all scripts
# read the same values from analysis_students.csv. Each variable is turned into
# a z-score within the classroom (mean 0, SD 1 among classmates), so it
# compares a child with her own classmates. Missing values stay missing.
#
#   attendance_z            balance check (Figure 1D, Table S16, Figures S15-S17)
#   gpa_z, age_z            correlates of Social Rank (Table S9, Figures S8, S30)
#   received_q1_z ... q9_z  nominations received in each question (same)
#   net_popularity_z        nominations received in Q4 minus Q5
#   aggression_z            mean of received_q6_z and received_q7_z (Table S9)
#   prosociality_z          mean of received_q8_z and received_q9_z (Table S9)
#   aggression_total_z      Q6 + Q7 nominations received, standardized (Figure S30)
#
# The balance regressions use GPA on its original 1-7 scale, as in the paper;
# gpa_z is used only in the correlates of Social Rank.
# Two computed ages above 15 years are birth-date errors (SI Table S1) and are
# set to missing before standardizing.

z_within <- function(x) (x - mean(x, na.rm = TRUE)) / sd(x, na.rm = TRUE)

nominations_received <- dyads %>%
  group_by(student_id = receiver_id) %>%
  summarise(across(starts_with("nom_q"), sum, .names = "received_{str_remove(.col, 'nom_')}"))

students <- students %>%
  left_join(nominations_received, by = "student_id") %>%
  mutate(age_clean = if_else(age > 15, NA_real_, age),
         net_popularity  = received_q4 - received_q5,
         aggression_total = received_q6 + received_q7) %>%
  group_by(class_id) %>%
  mutate(attendance_z = z_within(attendance),
         gpa_z        = z_within(gpa),
         age_z        = z_within(age_clean),
         across(c(starts_with("received_q"), net_popularity, aggression_total), z_within,
                .names = "{.col}_z"),
         aggression_z   = (received_q6_z + received_q7_z) / 2,
         prosociality_z = (received_q8_z + received_q9_z) / 2) %>%
  ungroup() %>%
  select(-age_clean)

cat("Within-classroom z-scores saved:\n")
## Within-classroom z-scores saved:
students %>%
  select(ends_with("_z")) %>%
  pivot_longer(everything(), names_to = "variable") %>%
  group_by(variable) %>%
  summarise(n = sum(!is.na(value)), mean = round(mean(value, na.rm = TRUE), 3),
            sd = round(sd(value, na.rm = TRUE), 3))

variable,n,mean,sd
age_z,853,0,0.974
aggression_total_z,859,0,0.974
aggression_z,859,0,0.926
attendance_z,859,0,0.974
gpa_z,859,0,0.974
net_popularity_z,859,0,0.974
prosociality_z,859,0,0.911
received_q1_z,859,0,0.974
received_q2_z,859,0,0.974
received_q3_z,859,0,0.974


# Step 7. Null-model filtering (robustness)

Some children give 5 tokens to everyone; others give 10 to some
classmates and 0 to others. Only the second kind of giving says
something about the partner. The filtered analysis asks whether the rank
pattern is still there when we look only at the transfers that depart
most from each child’s usual way of giving. It is a robustness check:
the main regressions already absorb each child’s average generosity with
sender fixed effects.

The null model shuffles, for each sender, *who* receives each of her
allocations, keeping the allocations themselves. Under this shuffle the
expected value and the standard deviation of any cell $i \to j$ are just
the mean and the standard deviation of the sender’s own allocations.
Each transfer is scored by

$$
d_{ij} = \frac{|w_{ij} - \bar{w}_i|}{\text{SD}(w_i)}.
$$

In [ ]:
# Null-model filtering (Methods and SI). For each sender we shuffle *who*
# receives each of her allocations, keeping all her allocations as they are.
# Under this shuffle every classmate is equally likely to get any of them, so
# the expected value and spread of any cell i -> j are simply the mean and the
# (population) standard deviation of sender i's own allocations.
# (The notebook checks this with 1,000 actual shuffles.)
#
# Each observed transfer gets a score: how far it is from the sender's usual
# giving, in units of the sender's own standard deviation.
#
#     d_ij = | tokens_sent_ij - mean_i | / sd_i

dyads <- dyads %>%
  group_by(sender_id) %>%
  mutate(sender_mean = mean(tokens_sent),
         sender_sd   = sqrt(mean((tokens_sent - sender_mean)^2)),
         departure   = if_else(sender_sd > 0,
                               abs(tokens_sent - sender_mean) / sender_sd,
                               0)) %>%
  ungroup()

**Check with actual shuffles.** The SI describes the null model with
1,000 random shuffles. Here we do them for one classroom and compare the
simulated mean and standard deviation of each cell with the closed form
used above.

In [ ]:
set.seed(2026)
one_class <- dyads %>% filter(class_id == 16)

simulated <- one_class %>%
  group_by(sender_id) %>%
  group_modify(function(rows, key) {
    shuffles <- replicate(1000, sample(rows$tokens_sent))   # one column per shuffle
    tibble(receiver_id = rows$receiver_id,
           tokens_sent = rows$tokens_sent,
           mean_sim    = rowMeans(shuffles),
           sd_sim      = apply(shuffles, 1, function(x) sqrt(mean((x - mean(x))^2))))
  }) %>%
  ungroup() %>%
  left_join(one_class %>% select(sender_id, receiver_id, sender_mean, sender_sd),
            by = c("sender_id", "receiver_id"))

simulated %>%
  summarise(max_difference_mean = max(abs(mean_sim - sender_mean)),
            max_difference_sd   = max(abs(sd_sim - sender_sd)))

max_difference_mean,max_difference_sd
0.22,0.0969793


The differences are of the size expected from 1,000 random draws (they
get smaller as the number of shuffles grows). The closed form is the
exact limit of the simulation, so we use it: same scores, and no
randomness in the results.

Then we keep the 50% of transfers with the highest score (one cutoff for
the whole sample). A pair enters the filtered Cooperation GAP only if
*both* directions are kept, so about a quarter of the pairs remain.

In [ ]:
# Keep the 50% of all directed transfers with the largest departure (one
# common cutoff for the whole sample). Dropped transfers become NA, not 0,
# because 0 is a real decision in the game.
# A dyad enters the filtered Cooperation GAP only if BOTH directions are kept.

retention <- 0.50
cutoff <- sort(dyads$departure, decreasing = TRUE)[ceiling(retention * nrow(dyads))]

reverse_departure <- dyads %>%
  select(class_id,
         sender_id   = receiver_id,
         receiver_id = sender_id,
         departure_reverse = departure)

dyads <- dyads %>%
  left_join(reverse_departure, by = c("class_id", "sender_id", "receiver_id")) %>%
  mutate(
    tokens_sent_filtered     = if_else(departure >= cutoff,         tokens_sent,     NA_real_),
    tokens_received_filtered = if_else(departure_reverse >= cutoff, tokens_received, NA_real_),
    coop_gap_filtered        = tokens_received_filtered - tokens_sent_filtered
  )

cat(sprintf("Share of transfers kept:                     %.3f\n",
            mean(!is.na(dyads$tokens_sent_filtered))))   # about 0.50
## Share of transfers kept:                     0.500
cat(sprintf("Share of decisions with both directions kept: %.3f\n",
            mean(!is.na(dyads$coop_gap_filtered))))      # about 0.26
## Share of decisions with both directions kept: 0.258

# Save

In [ ]:
dir.create(here("data", "generated"), recursive = TRUE, showWarnings = FALSE)
dir.create(here("output", "tables", "si"), recursive = TRUE, showWarnings = FALSE)
dir.create(here("output", "figures", "si"), recursive = TRUE, showWarnings = FALSE)

dyads <- dyads %>%
  select(class_id, round, sender_id, receiver_id,
         tokens_sent, tokens_received, coop_gap,
         friendship, sender_rank, receiver_rank, rank_gap,
         departure, tokens_sent_filtered, tokens_received_filtered, coop_gap_filtered,
         seconds_since_start, starts_with("nom_q"))

write_csv(dyads,    here("data", "generated", "analysis_dyads.csv"))
write_csv(students, here("data", "generated", "analysis_students.csv"))

The next notebook, `02_main_results.qmd`, uses these two files.

In [ ]:
sessionInfo()
## R version 4.3.2 (2023-10-31)
## Platform: aarch64-apple-darwin20 (64-bit)
## Running under: macOS 26.7.1
## 
## Matrix products: default
## BLAS:   /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRblas.0.dylib 
## LAPACK: /Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/lib/libRlapack.dylib;  LAPACK version 3.11.0
## 
## locale:
## [1] C.UTF-8/C.UTF-8/C.UTF-8/C/C.UTF-8/C.UTF-8
## 
## time zone: America/Santiago
## tzcode source: internal
## 
## attached base packages:
## [1] stats     graphics  grDevices utils     datasets  methods   base     
## 
## other attached packages:
##  [1] igraph_2.1.4    lubridate_1.9.4 forcats_1.0.0   stringr_1.5.1  
##  [5] dplyr_1.1.4     purrr_1.0.4     readr_2.1.5     tidyr_1.3.1    
##  [9] tibble_3.2.1    ggplot2_3.5.2   tidyverse_2.0.0 here_1.0.1     
## 
## loaded via a namespace (and not attached):
##  [1] bit_4.6.0         gtable_0.3.6      jsonlite_2.0.0    crayon_1.5.3     
##  [5] compiler_4.3.2    tidyselect_1.2.1  parallel_4.3.2    scales_1.3.0     
##  [9] yaml_2.3.10       fastmap_1.2.0     R6_2.6.1          labeling_0.4.3   
## [13] generics_0.1.3    knitr_1.50        munsell_0.5.1     rprojroot_2.0.4  
## [17] pillar_1.10.2     tzdb_0.5.0        rlang_1.1.6       stringi_1.8.7    
## [21] xfun_0.52         bit64_4.6.0-1     timechange_0.3.0  cli_3.6.4        
## [25] withr_3.0.2       magrittr_2.0.3    digest_0.6.37     grid_4.3.2       
## [29] vroom_1.6.5       hms_1.1.3         lifecycle_1.0.4   vctrs_0.6.5      
## [33] evaluate_1.0.3    glue_1.8.0        farver_2.1.2      colorspace_2.1-1 
## [37] rmarkdown_2.29    tools_4.3.2       pkgconfig_2.0.3   htmltools_0.5.8.1